In [ ]:
!pip install -U crewai

In [ ]:
!pip install "crewai[google-genai]"

#**Creative Game Designer Agent**
```
   role="Creative Game Designer",

   goal="Come up with fun, feasible game concepts and detailed mechanics based on user idea",
   
   backstory=
     "You are an experienced game designer."
     "You excel at turning vague ideas into clear, exciting game designs including:"
     "- core loop, rules, win/lose conditions"
     "- basic entities (player, enemies, items)"
     "- controls and feel"
     "Keep it simple enough to implement in pure Python + Pygame in one file.",
```

In [ ]:
from crewai import Agent,LLM
from google.colab import userdata

gemini_api_key = userdata.get('GEMINI_API_KEY')


llm = LLM(
   model="gemini/gemini-2.5-flash",
   api_key=gemini_api_key
)

game_designer = Agent(
    role="Creative Game Designer",
    goal="Come up with fun, feasible game concepts and detailed mechanics based on user idea",
    backstory=
      "You are an experienced game designer."
      "You excel at turning vague ideas into clear, exciting game designs including:"
      "- core loop, rules, win/lose conditions"
      "- basic entities (player, enemies, items)"
      "- controls and feel"
      "Keep it simple enough to implement in pure Python + Pygame in one file.",
    verbose=True,
    llm=llm,
)

In [ ]:
!pip install -U 'crewai[tools]'

#**Senior Python Game Developer Agent**

```
role="Senior Python Game Developer",

goal="Write clean, working Python code (using Pygame) for the described game",

backstory=
       "You are a senior software engineer specialized in Python game development with Pygame."
       "You write structured, readable code with:"
       "- Proper game loop, event handling, drawing"
       "- Comments explaining key parts"
       "- Error handling where needed"
       "You always produce a complete, runnable .py file.",
```

In [ ]:
from crewai_tools import SerperDevTool

serper_api_key = userdata.get('SERPER_API_KEY')

search_tool = SerperDevTool(api_key=serper_api_key)


senior_engineer = Agent(
   role="Senior Python Game Developer",
   goal="Write clean, working Python code (using Pygame) for the described game",
    backstory=
        "You are a senior software engineer specialized in Python game development with Pygame."
        "You write structured, readable code with:"
        "- Proper game loop, event handling, drawing"
        "- Comments explaining key parts"
        "- Error handling where needed"
        "You always produce a complete, runnable .py file.",
   verbose=True,
   llm=llm,
)


#**QA Engineer & Code Reviewer Agent**

    role="QA Engineer & Code Reviewer",

    goal="Test, review, and improve the code for bugs, playability, and completeness",
    
    backstory=
        "You are a meticulous QA engineer and code reviewer."
        "You carefully check:"
        "- Does the code run without errors?"
        "- Does it implement ALL the designed features?"
        "- Is it fun/playable? Any obvious balance issues?"
        "- Code style, variable names, comments"
        "Suggest fixes or small improvements and output the FINAL improved code.",


In [ ]:
qa_engineer = Agent(
    role="QA Engineer & Code Reviewer",
    goal="Test, review, and improve the code for bugs, playability, and completeness",
    backstory=
        "You are a meticulous QA engineer and code reviewer."
        "You carefully check:"
        "- Does the code run without errors?"
        "- Does it implement ALL the designed features?"
        "- Is it fun/playable? Any obvious balance issues?"
        "- Code style, variable names, comments"
        "Suggest fixes or small improvements and output the FINAL improved code.",
    verbose=True,
    llm=llm,
)


In [ ]:
from crewai import Task

In [ ]:
task_design = Task(
    description=
        "Take the user's game idea: {game_idea}"
        "1. Clarify and expand it into a fun, simple 2D game"
        "2. Describe: objective, controls, entities, win/lose"
        "3. Keep scope small (one level, basic mechanics)"
        "Output format:"
        "## Game Design Document"
        "- Title: ..."
        "- Genre: ..."
        "- Objective: ..."
        "- Controls: ..."
        "- Entities: ..."
        "- Mechanics: ...",
    expected_output="A clear markdown Game Design Document",
    agent=game_designer
)

In [ ]:
task_code = Task(
    description=
        "Using the game design from the previous task"
        "Write a COMPLETE, standalone Python script using Pygame that implements the game."
        "- Include import pygame, sys, random (if needed)"
        "- Full game loop, init, events, update, draw"
        "- Make it runnable with python game.py"
        "- Add simple comments"
        "- The main game loop must be exposed in the python code, it should not be inside any function like main"
        "- Final answer MUST be ONLY the Python code and Instructions on how to play the game",
    expected_output="A complete runnable Pygame Python script",
    agent=senior_engineer,
    context=[task_design]
)

In [ ]:
task_review = Task(
    description=
        "Review the Python code from the previous task."
        "1. Check for syntax/runtime errors"
        "2. Verify it matches the design document"
        "3. Test mentally: does it have init, loop, quit handling, drawing?"
        "4. Suggest fixes/improvements if needed"
        "5. Output the FINAL, improved, ready-to-run code"
        "Your final answer MUST be ONLY the complete Python code along with the instructions on how to play the game",
    expected_output="Final polished, runnable Pygame Python script and instructions on how to play the game",
    agent=qa_engineer,
    context=[task_design, task_code]
)

In [ ]:

from crewai import Crew, Process

game_crew = Crew(
      agents=[game_designer, senior_engineer, qa_engineer],
      tasks=[task_design, task_code, task_review],
      process=Process.sequential,
      verbose=True
)


In [ ]:
game_idea = "A fun endless runner where a character jumps over obstacles"
result = game_crew.kickoff(inputs={"game_idea": game_idea})
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  0bcd74a6-cd3d-43d3-87cb-dbc359f9bf9b                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Take the user's game idea: A fun endless runner where a character jumps over obstacles1. Clarify and     │
│  expand it into a fun, simple 2D game2. Describe: objective, controls, entities, win/lose3. Keep scope small    │
│  (one level, basic mechanics)Output format:## Game Design Document- Title: ...- Genre: ...- Objective: ...-     │
│  Controls: ...- Entities: ...- Mechanics: ...                                                                   │
│  ID: 8264631c-519c-47bb-b8b5-51a5070ace2f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Creative Game Designer                                                                                  │
│                                                                                                                 │
│  Task: Take the user's game idea: A fun endless runner where a character jumps over obstacles1. Clarify and     │
│  expand it into a fun, simple 2D game2. Describe: objective, controls, entities, win/lose3. Keep scope small    │
│  (one level, basic mechanics)Output format:## Game Design Document- Title: ...- Genre: ...- Objective: ...-     │
│  Controls: ...- Entities: ...- Mechanics: ...                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Creative Game Designer                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## Game Design Document                                                                                        │
│                                                                                                                 │
│  -   **Title:** Pixel Pounce                                                                                    │
│  -   **Genre:** Endless Runner, Arcade, 2D                                                                      │
│  -   **Objective:** Survive for as long as possible by jumping over obstacles and achieve the highest score.    │
│  -   **Controls:**                                                                                              │
│      *   `Spacebar` or `Up Arrow Key`: Jump.                                                                    │
│      *   `Spacebar` (on Game Over screen): Restart Game.                                                        │
│  -   **Entities:**                                                                                              │
│      *   **Player Character (The Sprinter):** A small, pixelated adventurer sprite (e.g., a simple block-man    │
│  or a tiny dinosaur). It automatically runs across the screen, maintaining a fixed horizontal position while    │
│  the game world scrolls towards it. The Sprinter has a "running" visual state (can be a simple animation or     │
│  just a static sprite) and a "jumping" visual state.                                                            │
│      *   **Ground (Endless Path):** A repeating horizontal strip of pixel art (e.g., grass, dirt, desert        │
│  floor) that scrolls continuously from right to left, creating the illusion of player movement.                 │
│      *   **Obstacles (Spiky Hazards):** Simple, ground-based static sprites such as small cacti, rocks, or      │
│  spike traps. These objects appear on the right edge of the screen and scroll left along with the ground.       │
│          *   For initial scope, we will have one primary type of obstacle. The challenge comes from their       │
│  placement and density.                                                                                         │
│      *   **Background (Distant Scenery):** Simple, non-interactive pixel art elements like distant mountains,   │
│  clouds, or trees that scroll slower than the foreground, adding a sense of depth.                              │
│  -   **Mechanics:**                                                                                             │
│      *   **Automatic Forward Movement:** The player character continuously "runs" forward. This is simulated    │
│  by having the game world (ground, obstacles, background) scroll from right to left at a constant, increasing   │
│  speed. The player's horizontal position remains fixed relative to the screen.                                  │
│      *   **Jumping:** Pressing the designated jump key makes the Sprinter perform a parabolic jump. The         │
│  Sprinter moves upwards, then downwards, automatically landing back on the ground. Only one jump is allowed at  │
│  a time (no double jumps).                                                                                      │
│      *   **Obstacle Generation:** New obstacles are randomly generated off-screen to the right. There will be   │
│  a minimum and maximum clear distance between obstacles

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Take the user's game idea: A fun endless runner where a character jumps over obstacles1. Clarify and expand    │
│  it into a fun, simple 2D game2. Describe: objective, controls, entities, win/lose3. Keep scope small (one      │
│  level, basic mechanics)Output format:## Game Design Document- Title: ...- Genre: ...- Objective: ...-          │
│  Controls: ...- Entities: ...- Mechanics: ...                                                                   │
│  Agent:                                                                                                         │
│  Creative Game Designer                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Review the Python code from the previous task.1. Check for syntax/runtime errors2. Verify it matches     │
│  the design document3. Test mentally: does it have init, loop, quit handling, drawing?4. Suggest                │
│  fixes/improvements if needed5. Output the FINAL, improved, ready-to-run codeYour final answer MUST be ONLY     │
│  the complete Python code along with the instructions on how to play the game                                   │
│  ID: ed6f4f1e-c34e-4222-a917-25471c7b9084                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: QA Engineer & Code Reviewer                                                                             │
│                                                                                                                 │
│  Task: Review the Python code from the previous task.1. Check for syntax/runtime errors2. Verify it matches     │
│  the design document3. Test mentally: does it have init, loop, quit handling, drawing?4. Suggest                │
│  fixes/improvements if needed5. Output the FINAL, improved, ready-to-run codeYour final answer MUST be ONLY     │
│  the complete Python code along with the instructions on how to play the game                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: QA Engineer & Code Reviewer                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Here's a comprehensive review of the "Pixel Pounce" Pygame script, followed by the improved, ready-to-run      │
│  code and instructions.                                                                                         │
│                                                                                                                 │
│  ### QA Review and Improvements:                                                                                │
│                                                                                                                 │
│  1.  **Syntax/Runtime Errors:**                                                                                 │
│      *   The code is syntactically correct and appears to run without immediate runtime errors.                 │
│      *   Iteration over `list(obstacles)` is correctly used to avoid `RuntimeError` when modifying the list     │
│  during iteration.                                                                                              │
│                                                                                                                 │
│  2.  **Verification against Design Document:**                                                                  │
│      *   **Title, Genre, Objective:** All matched.                                                              │
│      *   **Controls:** `Spacebar` and `Up Arrow` for jump, `Spacebar` to restart on Game Over. All matched.     │
│      *   **Entities:**                                                                                          │
│          *   **Player:** Blue block, fixed X, jumps. Matched.                                                   │
│          *   **Ground:** Scrolling brown and green strips. Matched, though the looping logic needs refinement   │
│  (addressed below).                                                                                             │
│          *   **Obstacles:** Dark green blocks, scroll left, one type. Matched.                                  │
│          *   **Background:** Grey mountains, white clouds, scroll slower. Matched, looping logic needs          │
│  refinement.                                                                                                    │
│      *   **Mechanics:**                                                                                         │
│          *   **Automatic Movement:** Game world scrolls. Matched.                                               │
│          *   **Jumping:** Parabolic, single jump. Matched.                                                      │
│          *   **Obstacle Generation:** Randomly generated, min/max gap. Matched, but frequency and density       │
│  scaling with speed can be improved (addressed below).                                                          │
│          *   **Collision Detection:** Bounding box with a small buffer. Matched.                                │
│          *   **Scoring System:** Survival points (1pt/100ms) and obstacle bonus (10pts). Matched.               │
│          *   **Progressive Difficulty:** Speed increases. Matched.                                              │
│          *   **Game Over/Restart:** Matched.                                                                    │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Review the Python code from the previous task.1. Check for syntax/runtime errors2. Verify it matches the       │
│  design document3. Test mentally: does it have init, loop, quit handling, drawing?4. Suggest                    │
│  fixes/improvements if needed5. Output the FINAL, improved, ready-to-run codeYour final answer MUST be ONLY     │
│  the complete Python code along with the instructions on how to play the game                                   │
│  Agent:                                                                                                         │
│  QA Engineer & Code Reviewer                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Here's a comprehensive review of the "Pixel Pounce" Pygame script, followed by the improved, ready-to-run code and instructions.

### QA Review and Improvements:

1.  **Syntax/Runtime Errors:**
    *   The code is syntactically correct and appears to run without immediate runtime errors.
    *   Iteration over `list(obstacles)` is correctly used to avoid `RuntimeError` when modifying the list during iteration.

2.  **Verification against Design Document:**
    *   **Title, Genre, Objective:** All matched.
    *   **Controls:** `Spacebar` and `Up Arrow` for jump, `Spacebar` to restart on Game Over. All matched.
    *   **Entities:**
        *   **Player:** Blue block, fixed X, jumps. Matched.
        *   **Ground:** Scrolling brown and green strips. Matched, though the looping logic needs refinement (addressed below).
        *   **Obstacles:** Dark green blocks, scroll left, one type. Matched.
        *   **Background:** Grey mountains, white clouds, scroll slower. Matched, looping log

In [ ]:
%%writefile main.py
### Paste your code here


Overwriting main.py


# **Do not edit any of the below code**

In [ ]:
# Install required packages
!pip install pygbag pyngrok -q

# **How to get pyngrok API Key?**
- Go to https://dashboard.ngrok.com/get-started/your-authtoken
- Copy your authtoken
- Run this cell with YOUR token:

In [ ]:
!ngrok authtoken 'Auth_token'

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
import subprocess
import time
from pyngrok import ngrok

# Step 1: Build the game
print("🔨 Building game...")
build_result = subprocess.run(
    ['pygbag', '--build', '--version', '0.9', '--PYBUILD', '3.12', '--cdn', 'https://pygame-web.github.io/archives/0.9/', 'main.py'],
    capture_output=True, text=True
)
print(build_result.stdout)

if build_result.returncode != 0:
    print("❌ Build failed:")
    print(build_result.stderr)
else:
    print("✅ Build successful!")

    # Step 2: Start HTTP server
    print("\n🚀 Starting server...")
    server = subprocess.Popen(
        ['python', '-m', 'http.server', '8000', '--directory', '/content/build/web'],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )

    # Wait for server to start
    time.sleep(5)

    # Step 3: Create ngrok tunnel
    try:
        print("🌐 Creating public URL...")
        public_url = ngrok.connect(8000)

        print("\n" + "="*60)
        print("🎮 YOUR GAME IS READY!")
        print("="*60)
        print(f"\n🔗 Click here to play: {public_url}")
        print("\n📝 How to play:")
        print("   • Press SPACE or Click to jump")
        print("="*60)

    except Exception as e:
        print(f"\n❌ Error creating tunnel: {e}")
        print("\n💡 Make sure you ran Cell 3 with a valid ngrok token")

🔨 Building game...

Serving python files from [/content/build/web]

with no security/performance in mind, i'm just a test tool : don't rely on me


SUMMARY
________________________

# the app folder
app_folder=/content

# artefacts directory
build_dir=/content/build/web

# cache directory
cache=/content/build/web-cache

# the window title and icon name
app_name=content

# package name, better make it unique
package=web.pygame.content-1770631226

# icons:  96x96 for desktop, 16x16 for web
icon=favicon.png

# js/wasm provider
cdn=https://pygame-web.github.io/archives/0.9/

now packing application ....


Now in /
     /main.py
Now in /sample_data
     /sample_data/README.md
     /sample_data/anscombe.json
     /sample_data/california_housing_test.csv
     /sample_data/california_housing_train.csv
     /sample_data/mnist_train_small.csv
     /sample_data/mnist_test.csv
optimizing /content
	/content : main.py
	/content : sample_data/README.md
	/content : sample_data/anscombe.json
	/content 